[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/01_kmeans/KMeans.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/clustering_methods/01_kmeans/KMeans.ipynb)

# Notebook 01 — K-Means Clustering
## Grouping Data Around Central Points

**Dataset**: Iris flowers (150 samples, 4 measurements)
**Question**: *Can we discover the 3 flower species without being told the labels?*
**Type**: Unsupervised — no target variable, algorithm discovers groups

---

## Section 1 — What Is K-Means?

### In plain English

Imagine you are sorting a pile of mixed flower samples into groups, but nobody told you
how many species there are or which sample belongs where.
You decide to use 3 representative "prototype" flowers:

1. Place 3 prototypes randomly in the pile
2. Assign every flower to the nearest prototype
3. Move each prototype to the centre of its assigned flowers
4. Repeat steps 2–3 until assignments stop changing

That is K-Means. The prototypes are called **centroids**.

> **You must specify K (number of clusters) in advance.**
> There are methods to choose K from the data (Elbow method, Silhouette score).

### Supervised vs Unsupervised

| | Supervised (classification) | **Unsupervised (clustering)** |
|---|---|---|
| Has labels | Yes — we know the answer | **No labels exist** |
| Goal | Predict the correct label | Discover natural groups |
| Evaluation | Accuracy, F1 | Silhouette score, inertia |
| Use case | Titanic survival | Customer segments, anomaly detection |

## Section 2 — How Does It Learn?

**Initialisation**: pick K centroids (K-Means++ picks them spread apart for stability)

**Repeat until convergence:**

**E-step** (Expectation) — assign each point to its nearest centroid:
```
cluster(x) = argmin_k  distance(x, centroid_k)
```

**M-step** (Maximisation) — move each centroid to the mean of its assigned points:
```
centroid_k = mean of all points assigned to cluster k
```

**Convergence**: when no point changes cluster between iterations (or max iterations reached)

### What K-Means assumes about cluster shape

K-Means works best when clusters are:
- **Roughly spherical** (centroid = mean makes sense)
- **Similar size**
- **Similar density**

It struggles with elongated, irregular, or very different-sized clusters.

## Section 3 — What Does the Data Need?

### Clustering is distance-based — scaling is essential

All clustering algorithms group points by how close they are to each other.
Distance is measured in feature space:
```
distance = sqrt((x1_a − x1_b)² + (x2_a − x2_b)² + ...)
```

If `sepal_length` ranges 4–8 cm and `petal_width` ranges 0.1–2.5 cm,
a 1 cm difference in sepal_length contributes equally to distance as a 1 cm difference in petal_width.
But the sepal feature has 4× the range — it dominates the distance calculation.

**Fix: StandardScaler** — all features end up with mean=0 and std=1.
Now every feature contributes equally to the distance.

### What clustering does NOT need

- No target labels (that is the whole point — we do not have them)
- No train/test split (there is no prediction to evaluate on held-out data)
- Only preparation: encode to numbers, fill missing values, scale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Load Iris — a classic dataset with 3 natural groups of flowers
df_raw = sns.load_dataset('iris')
FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
TRUE_LABELS = df_raw['species'].map({'setosa': 0, 'versicolor': 1, 'virginica': 2}).values

X_raw = df_raw[FEATURES].values

# Scale features — clustering algorithms are distance-based; scaling is required
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

print(f"Iris dataset: {X_raw.shape[0]} flowers, {X_raw.shape[1]} features")
print(f"True groups  : {df_raw['species'].value_counts().to_dict()}")
print()
print("We will PRETEND we do not know the species labels.")
print("The clustering algorithm must discover the 3 groups by itself.")
df_raw.head(6)

## Section 4 — Choose K, Train, and Visualise Clusters

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Elbow method — find the K where adding more clusters stops helping much
inertias    = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(K_range, inertias, 'o-', color='steelblue', lw=2)
ax1.set_xlabel('Number of clusters K', fontsize=11)
ax1.set_ylabel('Inertia (sum of squared distances to centroid)', fontsize=11)
ax1.set_title('Elbow Method\nLook for the bend — adding K past this gives little gain', fontsize=12)
ax1.axvline(3, color='orangered', ls='--', lw=1.5, label='K=3 (elbow)')
ax1.legend()

ax2.plot(K_range, silhouettes, 'o-', color='#26A69A', lw=2)
ax2.set_xlabel('Number of clusters K', fontsize=11)
ax2.set_ylabel('Silhouette score (higher = better separated)', fontsize=11)
ax2.set_title('Silhouette Score vs K\n1.0 = perfect separation, 0 = overlapping', fontsize=12)
ax2.axvline(3, color='orangered', ls='--', lw=1.5, label='K=3')
ax2.legend()

plt.tight_layout()
plt.show()
print(f"Best silhouette score at K=3: {silhouettes[1]:.4f}")

In [ ]:
# Train with K=3
km = KMeans(n_clusters=3, random_state=42, n_init=10)
pred_labels = km.fit_predict(X)

sil = silhouette_score(X, pred_labels)
print(f"Silhouette score (K=3): {sil:.4f}")
print()
print("Predicted cluster sizes:", dict(zip(*np.unique(pred_labels, return_counts=True))))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#5C6BC0', '#26A69A', '#FFA726']

for ax, (labels, title) in zip(axes, [
    (pred_labels, f'K-Means Clusters (K=3)\nSilhouette={sil:.3f}'),
    (TRUE_LABELS,  'True Species\n(setosa / versicolor / virginica)')
]):
    for k in range(3):
        mask = labels == k
        ax.scatter(X[mask, 2], X[mask, 3], color=colors[k], s=40, alpha=0.8,
                   label=f'Cluster {k}' if ax == axes[0] else ['setosa','versicolor','virginica'][k])
    ax.set_xlabel('petal_length (scaled)', fontsize=11)
    ax.set_ylabel('petal_width (scaled)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend()

plt.tight_layout()
plt.show()
print()
print('K-Means discovered the 3 species using only the measurements, with no labels.')
print('Compare left vs right — the discovered clusters align closely with the true species.')